# Letta / MemGPT 模式

> **把 LLM 的上下文窗口当作 RAM 来用：让智能体在三层记忆层级（核心记忆、召回记忆、归档记忆）之间主动换入换出信息，就像操作系统管理虚拟内存一样。**

LLM 的上下文窗口是有限的。一旦填满，旧信息会无声无息地消失。MemGPT 方法（Packer et al., 2023）把这个限制重新定义成一个**虚拟内存管理**问题。想想你电脑上的虚拟内存：你有一小块高速 RAM 和一块大而慢的硬盘。操作系统在两者之间交换数据，让程序能用上超过物理容量的内存。MemGPT 对智能体的上下文做了同样的事。

智能体维护三层显式记忆，并用**函数调用**（模型发出的、要求执行特定操作的指令）在层与层之间搬运数据。**内心独白**（用户永远看不到的私有推理）帮它决定记什么、归档什么、检索什么。**心跳机制**让多条记忆操作在一轮对话里串联执行。

本 notebook 采用**从零构建**的方式：
1. 只用 OpenAI SDK（调用 OpenAI 模型的 Python 库）实现一个最小 MemGPT 风格的智能体循环。不用任何框架，也不用 Letta 库。
2. 演示全部三层记忆、内心独白、心跳串联和记忆压力处理。
3. 最后展示 **Letta SDK** 如何把这些模式打包成生产级方案。

**读完本 notebook 你将：**
- 理解三层记忆模型（核心记忆、召回记忆、归档记忆）以及它为什么重要。
- 从零构建一个可运行的 MemGPT 风格智能体，拥有自编辑记忆。
- 看到内心独白和心跳控制流如何配合工作。
- 知道什么时候用从零构建的方案，什么时候用 Letta 平台。

## 关键概念

- **核心记忆（始终在上下文中）**：两个可编辑的文本块（persona 和 human），**始终包含**在系统提示词里。智能体通过 `core_memory_replace` 和 `core_memory_append` 函数调用重写这两个块，让它们保持最新。
- **召回记忆（对话历史）**：完整的历史消息，存储在上下文之外。当旧消息滑出上下文窗口后，智能体可以用 `recall_memory_search` 搜索它们，找回之前的对话轮次。
- **归档记忆（长期存储）**：一个可搜索的存储（由向量数据库或列表支撑），存放不属于核心记忆、但之后需要能找回来的信息。智能体用 `archival_memory_insert` 和 `archival_memory_search` 管理它。
- **内心独白**：在每条用户可见回复之前，智能体会先产生一个私有的「思考」步骤。它推理自己的记忆状态，决定存什么、取什么，并规划回复。这一步**不会展示给用户**。
- **心跳机制**：一次函数调用之后，智能体可以请求再执行一个处理步骤（一次「心跳」），在回复之前串联多个操作。这让单轮对话内就能完成多步记忆管理。
- **记忆压力**：当上下文窗口接近上限时，系统向智能体发出信号，提示它总结、归档或丢弃低优先级的信息。这能防止上下文悄悄溢出。
- **自编辑记忆**：本技术的标志性特性。随着对话演进，智能体自主重写自己的核心记忆块，让上下文中的知识始终保持准确和相关。

## 架构

<p align="center">
  <img src="../../images/diagrams/26_letta_memgpt_patterns.svg" alt="Letta/MemGPT 模式架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TB
    subgraph Context["LLM Context Window (limited)"]
        SYS["System prompt\n+ Core Memory blocks"]
        RECENT["Recent messages\n(sliding window)"]
        SYS --> RECENT
    end

    subgraph CoreMem["Core Memory (always in-context)"]
        P["Persona block\n'I am a helpful assistant\nwho remembers everything'"]
        H["Human block\n'Alice is a vegetarian\nwho works at Google'"]
    end

    subgraph RecallMem["Recall Memory (external)"]
        R[("Full conversation\nhistory\n─────────\nSearchable by\nkeyword / date")]
    end

    subgraph ArchivalMem["Archival Memory (external)"]
        A[("Long-term facts\n& documents\n─────────\nVector search\nor keyword")]
    end

    subgraph AgentLoop["Agent Loop"]
        IM["Inner monologue\n(private thinking)"]
        FC["Function calls:\ncore_memory_replace\ncore_memory_append\nrecall_memory_search\narchival_memory_insert\narchival_memory_search"]
        HB["Heartbeat\n(request more steps)"]
        RESP["User-visible response"]
    end

    CoreMem -->|"injected into"| SYS
    IM --> FC
    FC -->|"read/write"| CoreMem
    FC -->|"search"| RecallMem
    FC -->|"insert/search"| ArchivalMem
    FC -->|"request_heartbeat=true"| HB
    HB -->|"another turn"| IM
    IM --> RESP

    style P fill:#059669,color:#fff
    style H fill:#059669,color:#fff
    style R fill:#4f46e5,color:#fff
    style A fill:#7c3aed,color:#fff
    style IM fill:#d97706,color:#fff
    style FC fill:#dc2626,color:#fff
```

</details>

**每条用户消息触发的智能体循环：**
1. 系统提示词由当前核心记忆块 + 最近消息组装而成。
2. 智能体产生**内心独白**（私有推理）。
3. 智能体按需调用记忆函数（编辑核心记忆、搜索召回/归档记忆）。
4. 如果 `request_heartbeat=true`，循环从第 2 步重复，不需要用户输入。
5. 准备就绪后，智能体通过 `send_message` 发出**用户可见回复**。


In [ ]:
# 安装所需依赖（只需运行一次）
%pip install -q python-dotenv numpy deepagents langchain-openai

我们加载环境变量并创建 OpenAI 客户端。你的 `.env` 文件里需要有 `LLM_API_KEY`。`dotenv` 库会自动读取它。

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent
from langchain_core.messages import ToolMessage

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 核心实现 - 从零构建 MemGPT

我们分三部分构建一个完整的 MemGPT 风格智能体：

1. **记忆存储**：核心记忆、召回记忆、归档记忆及其操作。
2. **记忆工具**：智能体用来管理记忆的 OpenAI 函数调用工具。
3. **智能体循环**：带内心独白、心跳和记忆压力的主循环。

In [ ]:
class CoreMemory:
    """两个可编辑的文本块，始终包含在系统提示词中。

    - persona：描述智能体自己是谁
    - human：描述智能体对当前用户的了解
    """

    def __init__(self, persona: str = "", human: str = "", max_chars: int = 2000):
        self.persona = persona
        self.human = human
        self.max_chars = max_chars  # 每个块的上限

    def replace(self, block: str, old_text: str, new_text: str) -> str:
        """替换核心记忆块中的一个子串。"""
        current = getattr(self, block)
        if old_text not in current:
            return f"ERROR: '{old_text}' not found in {block} block."
        updated = current.replace(old_text, new_text, 1)
        if len(updated) > self.max_chars:
            return f"ERROR: replacement would exceed {self.max_chars} char limit."
        setattr(self, block, updated)
        return f"OK: {block} block updated."

    def append(self, block: str, text: str) -> str:
        """向核心记忆块追加文本。"""
        current = getattr(self, block)
        updated = current + text
        if len(updated) > self.max_chars:
            return f"ERROR: append would exceed {self.max_chars} char limit."
        setattr(self, block, updated)
        return f"OK: appended to {block} block."

    def format_for_prompt(self) -> str:
        return (
            f"<core_memory>\n"
            f"<persona>\n{self.persona}\n</persona>\n"
            f"<human>\n{self.human}\n</human>\n"
            f"</core_memory>"
        )

    def __repr__(self) -> str:
        return f"CoreMemory(persona={len(self.persona)} chars, human={len(self.human)} chars)"

`RecallMemory` 在上下文窗口之外保存完整对话历史。当旧消息滑出视野后，智能体按关键词搜索召回记忆，把它们找回来。

In [ ]:
class RecallMemory:
    """完整对话历史，存储在外部，可按关键词搜索。"""

    def __init__(self):
        self.messages: list[dict] = []

    def add(self, role: str, content: str) -> None:
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat(),
        })

    def search(self, query: str, limit: int = 5) -> list[dict]:
        """在对话历史上做关键词搜索。"""
        query_lower = query.lower()
        scored = []
        for msg in self.messages:
            text = msg["content"].lower()
            score = sum(1 for word in query_lower.split() if word in text)
            if score > 0:
                scored.append((score, msg))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [msg for _, msg in scored[:limit]]

    def __len__(self) -> int:
        return len(self.messages)

`ArchivalMemory` 是存放事实和文档的长期存储。它像一个可搜索的文件柜。智能体把重要信息存进来，之后用关键词搜索取回。

In [ ]:
class ArchivalMemory:
    """存放事实、文档和知识的长期可搜索存储。"""

    def __init__(self):
        self.entries: list[dict] = []

    def insert(self, content: str) -> str:
        self.entries.append({
            "content": content,
            "timestamp": datetime.now().isoformat(),
            "id": len(self.entries),
        })
        return f"OK: archived (id={len(self.entries) - 1}, total={len(self.entries)})."

    def search(self, query: str, limit: int = 5) -> list[dict]:
        """在归档条目上做关键词搜索。"""
        query_lower = query.lower()
        scored = []
        for entry in self.entries:
            text = entry["content"].lower()
            score = sum(1 for word in query_lower.split() if word in text)
            if score > 0:
                scored.append((score, entry))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [e for _, e in scored[:limit]]

    def __len__(self) -> int:
        return len(self.entries)


print("\u2713 CoreMemory, RecallMemory, ArchivalMemory defined")

接下来定义智能体可以调用的六个工具。它们是与 MemGPT 论文中记忆操作对应的 OpenAI 函数调用 schema。这些工具覆盖核心记忆编辑（`replace` 和 `append`）、召回与归档搜索、归档插入，以及 `send_message`（智能体与用户对话的唯一途径）。这一大块工具定义比较长。浏览每个工具的 `name` 和 `description` 字段就能明白它的用途。

我们把智能体的工具分成三组。第一组负责核心记忆编辑。这两个工具（`core_memory_replace` 和 `core_memory_append`）让智能体重写自己的 persona 块和 human 块。

`core_memory_replace` 让智能体替换核心记忆块中的一个子串。智能体用它纠正过时的事实（比如用户换了工作）。

In [ ]:
# -- 记忆编辑工具 --
# 它们让智能体重写自己的核心记忆块。

TOOLS_REPLACE = [
    {
        "type": "function",
        "function": {
            "name": "core_memory_replace",
            "description": (
                "Replace a substring in a core memory block. Use this to update "
                "outdated information about yourself (persona) or the user (human)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "block": {
                        "type": "string",
                        "enum": ["persona", "human"],
                        "description": "Which core memory block to edit.",
                    },
                    "old_text": {
                        "type": "string",
                        "description": "The exact substring to replace (must exist in the block).",
                    },
                    "new_text": {
                        "type": "string",
                        "description": "The replacement text.",
                    },
                },
                "required": ["block", "old_text", "new_text"],
            },
        },
    },
]

print(f"\u2713 {len(TOOLS_REPLACE)} core_memory_replace tool defined")


`core_memory_append` 向核心记忆块追加新文本。当智能体了解到关于用户或自己的新信息时就用它。我们把两个工具合并成 `TOOLS_EDIT`。

In [ ]:
# -- core_memory_append 工具 --

TOOLS_APPEND = [
    {
        "type": "function",
        "function": {
            "name": "core_memory_append",
            "description": (
                "Append new information to a core memory block. Use this to add "
                "newly learned facts about the user or update your persona."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "block": {
                        "type": "string",
                        "enum": ["persona", "human"],
                        "description": "Which core memory block to append to.",
                    },
                    "text": {
                        "type": "string",
                        "description": "The text to append.",
                    },
                },
                "required": ["block", "text"],
            },
        },
    },
]


# 合并编辑工具
TOOLS_EDIT = TOOLS_REPLACE + TOOLS_APPEND

print(f"\u2713 {len(TOOLS_EDIT)} memory editing tools combined")


第二组覆盖召回和归档操作。`recall_memory_search` 找回过去的对话轮次。`archival_memory_insert` 把事实长期保存。`archival_memory_search` 之后把它们检索出来。

`recall_memory_search` 找回可能已滑出上下文的过去对话轮次。`archival_memory_insert` 把重要事实长期保存。把这两个放在一起讲，因为它们都把智能体的记忆扩展到上下文窗口之外。

In [ ]:
# -- 搜索与归档工具 --
# 它们让智能体查过去的对话和已存的事实。

TOOLS_RECALL = [
    {
        "type": "function",
        "function": {
            "name": "recall_memory_search",
            "description": (
                "Search your conversation history (recall memory) for past messages. "
                "Use this when the user references something from earlier in the "
                "conversation that may have scrolled out of your current context."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query to find past messages.",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results to return (default 5).",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "archival_memory_insert",
            "description": (
                "Store information in long-term archival memory. Use this for facts, "
                "preferences, or knowledge that are important but don't need to be in "
                "your always-visible core memory."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string",
                        "description": "The information to archive.",
                    },
                },
                "required": ["content"],
            },
        },
    },
]

print(f"\u2713 {len(TOOLS_RECALL)} recall and archival-insert tools defined")


`archival_memory_search` 按关键词检索已存的知识。我们把所有搜索工具合并成一个 `TOOLS_SEARCH` 列表。

In [ ]:
# -- 归档搜索工具 --

TOOLS_ARCHIVAL_SEARCH = [
    {
        "type": "function",
        "function": {
            "name": "archival_memory_search",
            "description": (
                "Search your long-term archival memory for stored information."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query to find relevant archived information.",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results to return (default 5).",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
]


# 合并搜索工具
TOOLS_SEARCH = TOOLS_RECALL + TOOLS_ARCHIVAL_SEARCH

print(f"\u2713 {len(TOOLS_SEARCH)} search and archival tools combined")


第三组只有一个工具：`send_message`。这是智能体与用户对话的唯一途径。它写的其他所有内容都是私有的内心独白。我们把三组合并成一个 `MEMGPT_TOOLS` 列表。

In [ ]:
# -- 通信工具 --
# 这是智能体与用户对话的唯一途径。
# 智能体写的其他所有内容都是私有的内心独白。

TOOLS_COMM = [
    {
        "type": "function",
        "function": {
            "name": "send_message",
            "description": (
                "Send a visible message to the user. You MUST call this function "
                "to communicate with the user. Any text you generate that is NOT "
                "inside a send_message call is your private inner monologue."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "The message to send to the user.",
                    },
                },
                "required": ["message"],
            },
        },
    },
]

# 组装完整的工具列表
MEMGPT_TOOLS = TOOLS_EDIT + TOOLS_SEARCH + TOOLS_COMM

print(f"\u2713 {len(MEMGPT_TOOLS)} total memory tools assembled:")
for tool in MEMGPT_TOOLS:
    print(f"  \u2022 {tool['function']['name']}")


现在构建 `MemGPTAgent` 类。这是智能体的主循环。系统提示词模板把核心记忆和压力统计直接嵌进提示词，让智能体感知自己的记忆状态。构造函数初始化全部三层记忆，并配置上下文上限。

In [ ]:
class MemGPTAgent:
    """一个最小的 MemGPT 风格智能体：三层记忆加内心独白。

    实现核心的 MemGPT 循环：
    1. 用核心记忆组装系统提示词。
    2. 连同记忆工具一起发给 LLM。
    3. 处理工具调用（记忆操作）。
    4. 如果请求了心跳，回到第 2 步继续循环。
    5. 从 send_message 调用中提取用户可见回复。
    """

    SYSTEM_TEMPLATE = (
        "You are an AI assistant with a MemGPT-style memory system.\n"
        "\n"
        "## Your Memory System\n"
        "You have three types of memory:\n"
        "1. **Core memory** (below) -- always visible to you. Edit it to keep it current.\n"
        "2. **Recall memory** -- your full conversation history. Search it for past context.\n"
        "3. **Archival memory** -- long-term storage. Insert important facts; search when needed.\n"
        "\n"
        "## How to Communicate\n"
        "- Your raw text output is your PRIVATE inner monologue (the user cannot see it).\n"
        "- To send a message the user can see, you MUST call the `send_message` function.\n"
        "- Think before acting: use your inner monologue to reason about what you know,\n"
        "  what you need to remember, and what memory operations to perform.\n"
        "\n"
        "## Memory Management Rules\n"
        "- When the user shares personal information, update the `human` block in core memory.\n"
        "- When facts change (e.g., user moves cities), use `core_memory_replace` to update.\n"
        "- For detailed information that doesn't fit in core memory, use `archival_memory_insert`.\n"
        "- If the user asks about something from earlier, search recall or archival memory.\n"
        "- Keep core memory concise -- it's always in your context window.\n"
        "\n"
        "## Current Core Memory\n"
        "{core_memory}\n"
        "\n"
        "## Memory Stats\n"
        "- Recall memory: {recall_count} messages stored\n"
        "- Archival memory: {archival_count} entries stored\n"
        "- Context usage: {context_pct}% (memory pressure: {pressure_level})\n"
    )

构造函数初始化全部三层记忆并配置上限。`context_limit` 控制滑动窗口里保留多少条消息。`max_heartbeats` 限制每轮最多能做多少次工具调用循环。

In [ ]:
def __init__(
    self,
    model,
    persona: str = "I am a helpful assistant with persistent memory. I remember everything important about the people I talk to.",
    context_limit: int = 20,
    max_heartbeats: int = 5,
):
    self.model = model
    self.max_heartbeats = max_heartbeats
    self.context_limit = context_limit

    # 初始化三层记忆
    self.core = CoreMemory(persona=persona, human="(No information yet)")
    self.recall = RecallMemory()
    self.archival = ArchivalMemory()

    # 内部对话缓冲（用作上下文的滑动窗口）
    self._context_messages: list[dict] = []

    # 用于教学的日志
    self.turn_logs: list[dict] = []

MemGPTAgent.__init__ = __init__

接着给智能体加两个辅助方法。`_build_system_prompt` 用当前记忆内容和压力级别刷新系统消息。`_execute_tool` 把每个工具调用路由到对应的记忆操作，并返回结果字符串。

In [ ]:
def _build_system_prompt(self) -> str:
    """用当前核心记忆和统计信息组装系统提示词。"""
    context_pct = int(len(self._context_messages) / self.context_limit * 100)
    if context_pct > 80:
        pressure = "HIGH -- consider archiving information"
    elif context_pct > 50:
        pressure = "moderate"
    else:
        pressure = "low"

    return self.SYSTEM_TEMPLATE.format(
        core_memory=self.core.format_for_prompt(),
        recall_count=len(self.recall),
        archival_count=len(self.archival),
        context_pct=context_pct,
        pressure_level=pressure,
    )

def _execute_tool(self, name: str, args: dict) -> str:
    """执行一个记忆工具并返回结果字符串。"""
    if name == "core_memory_replace":
        return self.core.replace(args["block"], args["old_text"], args["new_text"])
    elif name == "core_memory_append":
        return self.core.append(args["block"], args["text"])
    elif name == "recall_memory_search":
        results = self.recall.search(args["query"], args.get("limit", 5))
        if not results:
            return "No matching messages found in recall memory."
        return "\n".join(
            f"[{r['timestamp']}] {r['role']}: {r['content']}" for r in results
        )
    elif name == "archival_memory_insert":
        return self.archival.insert(args["content"])
    elif name == "archival_memory_search":
        results = self.archival.search(args["query"], args.get("limit", 5))
        if not results:
            return "No matching entries found in archival memory."
        return "\n".join(f"[{r['id']}] {r['content']}" for r in results)
    elif name == "send_message":
        return "Message sent."
    else:
        return f"ERROR: Unknown tool '{name}'."

MemGPTAgent._build_system_prompt = _build_system_prompt
MemGPTAgent._execute_tool = _execute_tool

`chat` 方法是 MemGPT 循环的心脏。它把用户消息存进召回记忆，修剪上下文窗口，然后进入 while 循环。循环内部智能体私有推理（内心独白）、调用记忆工具，并可请求心跳进入下一轮。当 `send_message` 被调用或心跳用尽时，循环结束。

`chat` 方法是 MemGPT 循环的心脏。它把用户消息存进召回记忆，修剪上下文窗口，然后进入 while 循环。循环内部智能体私有推理（内心独白）、调用记忆工具，并可请求心跳进入下一轮。当 `send_message` 被调用或心跳用尽时，循环结束。

`chat` 方法搭建上下文并委托给内部循环。它把用户消息存进召回记忆，修剪旧消息，并准备好跟踪回复、工具调用和心跳的状态字典。

In [ ]:
def chat(self, user_message: str, verbose: bool = False) -> str:
    """让用户消息走一遍 MemGPT 循环。"""
    # 存入召回记忆
    self.recall.add("user", user_message)
    self._context_messages.append({"role": "user", "content": user_message})

    # 按需修剪上下文窗口
    if len(self._context_messages) > self.context_limit:
        self._context_messages = self._context_messages[-self.context_limit:]

    state = {
        "user_response_parts": [],
        "inner_monologue_parts": [],
        "tool_calls_log": [],
        "heartbeat_count": 0,
    }

    messages = [{"role": "system", "content": self._build_system_prompt()}]
    messages.extend(self._context_messages)

    self._run_loop(messages, state, verbose)

    return self._finalize_turn(
        user_message, state["user_response_parts"],
        state["inner_monologue_parts"], state["tool_calls_log"],
        state["heartbeat_count"],
    )

MemGPTAgent.chat = chat


`_run_loop` 是 MemGPT 的核心循环。每次迭代把上下文发给 LLM，捕获内心独白，处理工具调用，并决定是心跳（再来一轮）还是停止。当 `send_message` 被调用或心跳用尽时，循环结束。

In [ ]:
def _run_loop(self, messages: list, state: dict, verbose: bool) -> None:
    """MemGPT 内部循环：推理、调用工具、心跳、重复。"""
    model_with_tools = self.model.bind_tools(MEMGPT_TOOLS)

    while state["heartbeat_count"] <= self.max_heartbeats:
        response = model_with_tools.invoke(messages)
        assistant_msg = response

        inner = to_text(assistant_msg.content)
        if inner:
            state["inner_monologue_parts"].append(inner)
            if verbose:
                print(f"  💭 Inner monologue: {inner[:120]}...")

        if not assistant_msg.tool_calls:
            break

        messages.append(assistant_msg)

        request_heartbeat = False
        for tc in assistant_msg.tool_calls:
            fn_name = tc["name"]
            fn_args = tc["args"]

            if verbose:
                print(f"  🔧 Tool call: {fn_name}({json.dumps(fn_args, ensure_ascii=False)[:100]})")

            result = self._execute_tool(fn_name, fn_args)
            state["tool_calls_log"].append({"name": fn_name, "args": fn_args, "result": result})

            if fn_name == "send_message":
                state["user_response_parts"].append(fn_args["message"])

            messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

            if fn_name != "send_message":
                request_heartbeat = True

        if request_heartbeat and not state["user_response_parts"]:
            state["heartbeat_count"] += 1
            if verbose:
                print(f"  ❤️ Heartbeat #{state['heartbeat_count']}")
            continue
        else:
            break

MemGPTAgent._run_loop = _run_loop


`_finalize_turn` 组装用户可见回复，存入召回记忆，并记录完整轨迹方便调试。把它从主循环里拆出来，让每个单元格职责单一。

In [ ]:
def _finalize_turn(
    self, user_message, user_response_parts,
    inner_monologue_parts, tool_calls_log, heartbeat_count,
) -> str:
    """收尾一轮：组装回复、存储并记录轨迹。"""
    final_response = "\n".join(user_response_parts) if user_response_parts else "(No response generated)"

    # 把助手回复存入召回记忆
    self.recall.add("assistant", final_response)
    self._context_messages.append({"role": "assistant", "content": final_response})

    # 记录日志供检查
    self.turn_logs.append({
        "user_message": user_message,
        "inner_monologue": inner_monologue_parts,
        "tool_calls": tool_calls_log,
        "response": final_response,
        "heartbeats": heartbeat_count,
    })

    return final_response

MemGPTAgent._finalize_turn = _finalize_turn


这两个检查方法让你看到智能体记忆和推理的内部。`inspect_memory` 打印全部三层记忆。`inspect_last_turn` 展示最近一轮的内心独白、工具调用和心跳次数。

In [ ]:
def inspect_memory(self) -> None:
    """打印三层记忆的当前状态。"""
    print("=" * 60)
    print("MEMORY STATE")
    print("=" * 60)
    print(f"\n--- Core Memory ---")
    print(f"Persona: {self.core.persona}")
    print(f"Human:   {self.core.human}")
    print(f"\n--- Recall Memory ({len(self.recall)} messages) ---")
    for msg in self.recall.messages[-5:]:
        print(f"  [{msg['role']}] {msg['content'][:80]}...")
    if len(self.recall) > 5:
        print(f"  ... and {len(self.recall) - 5} earlier messages")
    print(f"\n--- Archival Memory ({len(self.archival)} entries) ---")
    for entry in self.archival.entries:
        print(f"  [{entry['id']}] {entry['content'][:80]}...")
    if not self.archival.entries:
        print("  (empty)")

def inspect_last_turn(self) -> None:
    """展示最近一轮智能体的详细轨迹。"""
    if not self.turn_logs:
        print("No turns yet.")
        return
    turn = self.turn_logs[-1]
    print("=" * 60)
    print("LAST TURN TRACE")
    print("=" * 60)
    print(f"\nUser: {turn['user_message']}")
    if turn["inner_monologue"]:
        print(f"\n\U0001f4ad Inner monologue:")
        for m in turn["inner_monologue"]:
            print(f"   {m[:200]}")
    print(f"\n\U0001f527 Tool calls ({len(turn['tool_calls'])}):")
    for tc in turn["tool_calls"]:
        args_str = json.dumps(tc["args"], ensure_ascii=False)
        print(f"   {tc['name']}({args_str[:100]})")
        print(f"   \u2192 {tc['result'][:100]}")
    print(f"\n\u2764\ufe0f Heartbeats: {turn['heartbeats']}")
    print(f"\nAgent: {turn['response']}")

MemGPTAgent.inspect_memory = inspect_memory
MemGPTAgent.inspect_last_turn = inspect_last_turn

print("\u2713 MemGPTAgent class defined")

## 使用示例 - 自编辑记忆实战

我们走一遍多轮对话，智能体会：
1. 了解用户并更新核心记忆。
2. 把放不进核心记忆的详细信息归档。
3. 在被问到时取回归档信息。
4. 通过替换过时事实来处理矛盾。

In [ ]:
# 创建一个智能体
agent = MemGPTAgent(model=llm)

# 第 1 轮：用户自我介绍
print("=" * 60)
print("TURN 1 \u2014 Introduction")
print("=" * 60)
response = agent.chat(
    "Hi! I'm Alice. I'm a data scientist at Google, and I live in San Francisco. "
    "I'm a vegetarian and my favorite cuisine is Italian.",
    verbose=True,
)
print(f"\nAgent: {response}")

看看智能体存了什么。`inspect_memory` 方法打印全部三层记忆，让你看到智能体从自我介绍中记住了什么。

In [ ]:
# 看看智能体在记忆里存了什么
agent.inspect_memory()

这一轮用户分享了某个 ML 项目的详细技术信息。观察智能体是否会把它归档到长期记忆。这些信息对核心记忆来说太细，但又重要到不能丢。

In [ ]:
# 第 2 轮：用户分享详细技术信息
print("=" * 60)
print("TURN 2 \u2014 Detailed information (archive candidate)")
print("=" * 60)
response = agent.chat(
    "I'm working on a project using PyTorch to build a recommendation system. "
    "The model architecture uses a two-tower approach with a user encoder and "
    "an item encoder, both using transformer blocks. We're training on 50M "
    "user-item interactions from the last 6 months. The key challenge is "
    "handling cold-start users who have fewer than 5 interactions.",
    verbose=True,
)
print(f"\nAgent: {response}")

现在用户的事实变了：新工作和新城市。这是对自编辑记忆的关键检验。智能体应该用 `core_memory_replace` 更新过时信息，而不是追加相互矛盾的事实。

In [ ]:
# 第 3 轮：用户的事实变了 \u2014 智能体应更新核心记忆
print("=" * 60)
print("TURN 3 \u2014 Fact change (self-editing memory)")
print("=" * 60)
response = agent.chat(
    "Big news \u2014 I just accepted a new position! I'm leaving Google "
    "and joining OpenAI next month as a research scientist. Also, "
    "I'm moving to London for the role.",
    verbose=True,
)
print(f"\nAgent: {response}")

验证智能体是否更新了核心记忆。human 块现在应该显示新工作（OpenAI）和新城市（伦敦），而不是旧值。

In [ ]:
# 检查核心记忆是否已用新事实更新
agent.inspect_memory()

print("\n\u2713 The agent updated Alice's job and location in core memory!")

用户问起第 2 轮的 ML 项目。如果智能体归档了那部分信息，它应该搜索归档记忆并取回细节。留意 `archival_memory_search` 工具调用。

In [ ]:
# 第 4 轮：用户问起之前的事
print("=" * 60)
print("TURN 4 \u2014 Archival memory retrieval")
print("=" * 60)
response = agent.chat(
    "What do you remember about the ML project I was working on?",
    verbose=True,
)
print(f"\nAgent: {response}")

我们来追踪智能体在最后一轮的推理。这里会展示内心独白、调用了哪些工具、用了多少次心跳。这种可观测性是 MemGPT 调试时的一大优势。

In [ ]:
# 查看最后一轮的详细轨迹
# 这里展示内心独白、工具调用和心跳
agent.inspect_last_turn()

## 记忆压力与上下文管理

当上下文窗口被填满时，智能体会收到压力信号。它应该主动归档或总结信息。我们用一个小上下文上限来模拟这个过程，观察智能体如何管理自己的上下文。

In [ ]:
# 创建一个小上下文窗口的智能体，快速触发压力
small_agent = MemGPTAgent(model=llm, context_limit=8)

# 连环发送消息填满上下文
messages = [
    "My favorite color is blue and I enjoy hiking on weekends.",
    "I have two cats named Luna and Mochi.",
    "I'm learning Rust and really enjoying the borrow checker.",
    "My birthday is March 15th and I love chocolate cake.",
    "I'm training for a half marathon in October.",
    "I just started reading 'Designing Data-Intensive Applications'.",
    "My partner's name is Sam and we've been together for 3 years.",
]

for i, msg in enumerate(messages, 1):
    sep = '=' * 50
    print(f"\n{sep}")
    print(f"Turn {i} (context: {len(small_agent._context_messages)}/{small_agent.context_limit})")
    print(f"{sep}")
    print(f"User: {msg}")
    response = small_agent.chat(msg, verbose=True)
    print(f"Agent: {response[:150]}...")

# 最终状态
print("\n" + "=" * 50)
print("FINAL MEMORY STATE")
print("=" * 50)
small_agent.inspect_memory()

## 心跳串联 - 一轮内完成多个操作

心跳机制让智能体在回复之前执行多条记忆操作。当一条用户消息既要读记忆又要写记忆时，这一点至关重要。

In [ ]:
# 用于心跳演示的新智能体
hb_agent = MemGPTAgent(model=llm)

# 预先往归档记忆里放几条事实
hb_agent.archival.insert("Alice's favorite restaurant is Chez Panisse in Berkeley.")
hb_agent.archival.insert("Alice is allergic to shellfish but loves sushi without shrimp.")
hb_agent.archival.insert("Alice prefers window seats and flies Delta.")

print("Pre-loaded archival memory:")
for e in hb_agent.archival.entries:
    print(f"  [{e['id']}] {e['content']}")

# 问一个需要搜索归档 + 更新核心记忆 + 回复的问题
print("\n" + "=" * 60)
print("MULTI-STEP HEARTBEAT TURN")
print("=" * 60)
response = hb_agent.chat(
    "I'm planning a dinner out. What do you know about my food preferences "
    "and restaurant choices? Also, I should mention I'm no longer allergic "
    "to shellfish \u2014 turns out it was a misdiagnosis!",
    verbose=True,
)
print(f"\nAgent: {response}")

# 展示心跳链
print(f"\nHeartbeats used: {hb_agent.turn_logs[-1]['heartbeats']}")
print(f"Tool calls in this turn: {len(hb_agent.turn_logs[-1]['tool_calls'])}")
for tc in hb_agent.turn_logs[-1]["tool_calls"]:
    print(f"  \u2022 {tc['name']}")

注意：`http://localhost:8283` 是用 `letta server start` 在本地运行 Letta 服务时的默认地址。它不是公网 URL；生产环境使用时请换成你自己的 Letta 部署主机。

## 生产环境替代方案 - Letta SDK

上面的从零实现用来说明 MemGPT 的思路。生产环境中，**Letta** 平台以现成的基础设施提供同样的模式：

- **持久化存储**：核心记忆、召回记忆和归档记忆由数据库支撑（PostgreSQL + pgvector）。
- **多智能体支持**：多个智能体可共享或隔离记忆。
- **REST API**：通过完整的 API 把智能体部署成服务。
- **预置工具**：记忆管理工具、网页搜索、代码执行等。
- **托管服务**：Letta Cloud 免运维部署。

下面演示如何创建一个 Letta 智能体并与之交互。这需要运行一个 Letta 服务（`letta server` 或 Docker）。

```python
# pip install letta

from letta import create_client

# Connect to a running Letta server (or use Letta Cloud)
client = create_client()  # defaults to http://localhost:8283

# Create an agent with custom persona and human blocks
agent_state = client.create_agent(
    name="my_assistant",
    memory=ChatMemory(
        persona="I am a helpful research assistant specializing in ML papers.",
        human="User has not introduced themselves yet.",
    ),
    llm="openai/gpt-4o-mini",
    embedding="openai/text-embedding-3-small",
)

# Chat - the agent manages its own memory automatically
response = client.send_message(
    agent_id=agent_state.id,
    role="user",
    message="Hi! I'm working on a paper about attention mechanisms.",
)

# The response includes both inner monologue and user-visible messages
for msg in response.messages:
    if msg.message_type == "internal_monologue":
        print(f"[thinking] {msg.internal_monologue}")
    elif msg.message_type == "assistant_message":
        print(f"[response] {msg.assistant_message}")

# Inspect the agent's memory
memory = client.get_in_context_messages(agent_id=agent_state.id)
core = client.get_core_memory(agent_id=agent_state.id)
print(f"Persona block: {core.get_block('persona').value}")
print(f"Human block: {core.get_block('human').value}")

# Search archival memory
results = client.get_archival_memory(agent_id=agent_state.id, query="attention")
```

Letta SDK 替我们处理了所有手动搭建的管道。系统提示词组装、工具执行、心跳循环、记忆持久化、上下文管理都由它负责。你只需专注于智能体的行为。

## 讨论与权衡

### 优点

- **显式的记忆管理**：与不透明的「记忆」层不同，智能体的记忆操作可见、可审计、可调试。你可以精确检查智能体选择记住了什么、忘掉了什么、更新了什么。
- **优雅的上下文溢出**：上下文填满时，旧信息不会悄悄丢失；智能体主动管理什么留在上下文、什么被归档。信息不会真正丢失。
- **自编辑记忆**：智能体重写自己的核心记忆块，让它们保持最新。事实变化时（用户搬家、换工作），智能体做更新，而不是堆积过时信息。
- **内心独白**：私有的推理通道让智能体「先想后说」。它在不打扰用户视野的情况下规划记忆操作和回复。
- **心跳串联**：多步记忆操作在单轮内顺畅完成。用户无需等待，也无需补发消息。

### 缺点

- **token 开销大**：记忆管理工具、系统提示词和核心记忆块占用大量上下文空间。对话很短时，这些开销可能不值得。
- **LLM 可靠性**：智能体必须正确决定*何时*以及*如何*使用记忆工具。LLM 可能忘记更新核心记忆、替换出错，或者把不重要的信息过度归档。
- **关键词搜索的局限**：我们的从零实现用关键词匹配。生产系统需要向量检索（用嵌入按含义找相似内容）才能可靠地做语义检索。
- **复杂度成本**：完整的 MemGPT 循环（内心独白 + 工具调用 + 心跳）增加延迟和成本。每次心跳都是一次额外的 LLM 调用。
- **冷启动**：新智能体的核心记忆很少，也没有归档知识。要积累出有用的上下文需要好几轮。

### 何时使用 MemGPT 模式

| 场景 | 建议 |
|----------|---------------|
| 长期运行的个人助理（数百个会话） | **非常好**：核心使用场景 |
| 智能体需要维护准确、持续演变的用户画像 | **非常好**：自编辑核心记忆在这里最有价值 |
| 短对话聊天机器人 | **大材小用**：滑动窗口或摘要记忆就够了 |
| 智能体管理复杂、分层的知识 | **良好**：用归档记忆承载深度 |
| 成本敏感、高吞吐的应用 | **考虑替代方案**：心跳和内心独白会增加 API 调用 |
| 调试智能体行为 | **非常好**：内心独白提供完整推理轨迹 |

### MemGPT 与其他记忆模式的对比

| 维度 | MemGPT / Letta | Mem0 | 摘要记忆 | RAG |
|--------|---------------|------|----------------|-----|
| 记忆模型 | 三层（核心 + 召回 + 归档） | 扁平键值存储 | 压缩摘要 | 外部文档 |
| 谁管理记忆 | 智能体自己 | 库（自动） | 系统（周期性） | 开发者 |
| 自编辑 | 有（重写核心记忆） | 有（冲突处理） | 无 | 无 |
| 内心独白 | 有 | 无 | 无 | 无 |
| 上下文溢出处理 | 显式（归档换页） | 未处理 | 摘要 | 检索 |
| 最适合 | 长期运行的智能体 | 快速个性化 | 低成本压缩 | 知识密集型应用 |

## 延伸阅读

- [Packer et al., "MemGPT: Towards LLMs as Operating Systems" (2023)](https://arxiv.org/abs/2310.08560) - 提出 LLM 虚拟内存方法的原始论文
- [Letta 文档](https://docs.letta.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 生产级 MemGPT 平台的官方文档
- [Letta GitHub 仓库](https://github.com/letta-ai/letta) - 开源代码、示例和社区
- [PyPI 上的 Letta Python SDK](https://pypi.org/project/letta/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 包详情与安装方式
- [OpenAI 函数调用指南](https://platform.openai.com/docs/guides/function-calling?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 记忆工具调用使用的 API 机制
- [Anthropic - Building Effective Agents (2025)](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 与记忆管理互补的智能体架构模式
- [Mem0 - 自动记忆层](https://github.com/mem0ai/mem0) - 替代方案：无需智能体自管理的自动记忆抽取

---

*← 上一篇：[25 - Mem0 集成模式](../25_mem0_patterns/) · 下一篇：[27 - Zep 记忆](../27_zep_memory/) →*

## 🧪 动手试一试

三个小挑战，加深理解。每个大约需要 10-30 分钟。

### 挑战 1：第三个核心记忆块
在 `CoreMemory` 的 persona 和 human 旁边加一个「project」块。修改 `MemGPTAgent`，把新块加进系统提示词。对话中用 `core_memory_replace` 写入项目细节，并验证它们能跨轮次保留。

### 挑战 2：心跳链分析
对一个需要多次记忆操作的复杂查询运行 `chat()`（比如「先保存这个，再搜索相关内容，然后回答」）。统计 `_run_loop()` 在调用 `send_message` 之前经历了多少次心跳迭代。用 5 个这样的查询测试，报告平均链深。

### 挑战 3：归档检索策略
往 `ArchivalMemory` 里放 20 条事实。对比基于关键词的 `archival_memory_search` 和基于嵌入余弦相似度的搜索。对每种策略在 10 个测试查询上测量查全率。这与 20 记忆检索模式中的检索流程思路相通。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--26-letta-memgpt-patterns--letta-memgpt-patterns)
